In [0]:
import time

TABLE = "retailmart.sales_transactions"
DASHBOARD_LOOKBACK_DAYS = 90
PATTERN_LOOKBACK_DAYS = 30


def print_section(title: str) -> None:
    print("\n" + "=" * 50)
    print(title)
    print("=" * 50)


print_section("INVESTIGATION 1: Benchmark Sam's slow query")

sam_query = f"""
    SELECT
        city,
        product_category,
        COUNT(*) AS total_orders,
        SUM(amount) AS total_revenue,
        AVG(amount) AS avg_order_value
    FROM {TABLE}
    WHERE sale_date >= date_sub(current_date(), {DASHBOARD_LOOKBACK_DAYS})
      AND city = 'New York'
      AND product_category = 'Electronics'
    GROUP BY city, product_category
    ORDER BY total_revenue DESC
"""

start = time.time()
df_sam_query = spark.sql(sam_query)
df_sam_query.show()
query_time_1 = round(time.time() - start, 2)
print(f"\nQuery time: {query_time_1}s")

print_section("INVESTIGATION 2: Table health check")

table_detail_df = spark.sql(f"DESCRIBE DETAIL {TABLE}")
table_detail_df.select("numFiles", "sizeInBytes", "partitionColumns").show(truncate=False)

print_section("INVESTIGATION 3: Check optimization history")

spark.sql(f"DESCRIBE HISTORY {TABLE}") \
    .select("version", "timestamp", "operation", "operationParameters") \
    .show(20, truncate=False)

print_section("INVESTIGATION 4: Query execution plan")
df_sam_query.explain(mode="formatted")

print_section("INVESTIGATION 5: Query pattern analysis")

queries = {
    "Filter by City only": "WHERE city = 'Chicago'",
    "Filter by Category only": "WHERE product_category = 'Electronics'",
    "Filter by Date only": f"WHERE sale_date >= date_sub(current_date(), {PATTERN_LOOKBACK_DAYS})",
    "Filter by City + Category": "WHERE city = 'Chicago' AND product_category = 'Electronics'",
    "Filter by Date + City": f"WHERE sale_date >= date_sub(current_date(), {PATTERN_LOOKBACK_DAYS}) AND city = 'New York'",
}

print(f"\n{'Query Pattern':<35} {'Time (s)':>10}")
print("-" * 50)

query_times = {}
for pattern, where_clause in queries.items():
    t_start = time.time()
    spark.sql(
        f"""
        SELECT COUNT(*) AS total_orders, SUM(amount) AS total_revenue
        FROM {TABLE}
        {where_clause}
        """
    ).collect()
    elapsed = round(time.time() - t_start, 2)
    query_times[pattern] = elapsed
    print(f"{pattern:<35} {elapsed:>10}s")

print("\nInvestigation complete. Fill in the findings template in the markdown cell below.")


## Findings template

Use this section to capture the final investigation summary after reviewing the outputs above.

### TICKET-1001 Investigation Findings

**Investigated by:** [YOUR NAME]  
**Date:** [TODAY'S DATE]  
**Table:** `retailmart.sales_transactions`

#### Problem
* Sam's dashboard query takes: _____ seconds
* Expected: < 5 seconds

#### Root cause
* Number of files in table: _____
* Partition columns: _____
* Z-ordering applied: YES / NO
* OPTIMIZE ever run: YES / NO

#### What needs to be fixed
* [ ] Apply `OPTIMIZE`
* [ ] Apply `ZORDER BY (_____, _____, _____)`
* [ ] Schedule a daily optimization job

#### Expected improvement
* Estimated query time after fix: _____ seconds

#### Columns to ZORDER BY
1. ___________ (most filtered)
2. ___________ (second most filtered)
3. ___________ (third most filtered)

#### Latest run snapshot
* Query time observed: 0.48s
* Table files observed: 10
* Partition columns observed: `[year]`
* OPTIMIZE found in history: YES
* Z-order columns shown in recent history rows: none
